# Webcam: CNN + motion detection

Press **Run All**. The live camera feed appears under the final cell. Keep the camera still, with the view clear for its first 30 frames. Use **Stop/Interrupt** to finish.

Each moving region has a cross marking its bounding-box center and an **X / Y** pixel readout. Coordinates refer to the resized preview (640 pixels wide), with **(0, 0) at the top-left**, X increasing rightward and Y downward. These are image positions, not real-world distances.


## 1. Imports and paths


In [ ]:
from pathlib import Path
import sys
import cv2
from PIL import Image
import torch
from torchvision import datasets

PROJECT = Path('.')
DATA = Path('data/missile_dataset')
sys.path.insert(0, str(PROJECT))


## 2. Load the trained CNN


In [ ]:
# The actual CNN layers are in cnn.py beside this notebook.
from cnn import load_model
model, transform, class_names = load_model(PROJECT / "missile_model.pth")

def predict(inputs):
    with torch.inference_mode():
        return model(inputs).softmax(1)


## 3. Test accuracy


In [ ]:
from cnn import classify_probabilities

test_data = datasets.ImageFolder(DATA / "test", transform=transform)
assert test_data.classes == class_names, "Dataset/model class order differs"
test_loader = torch.utils.data.DataLoader(test_data, batch_size=16)
correct, total, decided, accepted_correct = 0, 0, 0, 0
for images, labels in test_loader:
    probabilities = predict(images)
    predictions = probabilities.argmax(1)
    correct += (predictions == labels).sum().item()
    total += len(labels)
    outputs = classify_probabilities(probabilities, class_names, model.inference_policy)
    for (name, score), expected in zip(outputs, labels):
        if name != "uncertain":
            decided += 1
            accepted_correct += name == class_names[int(expected)]
print(f"Raw CNN accuracy on original test photos: {100 * correct / max(1, total):.2f}% ({correct}/{total})")
print(f"Default policy: {decided}/{total} classified, {total-decided} uncertain")
print(f"Accuracy among classified photos: {100 * accepted_correct / max(1, decided):.2f}%")
print("New hand/tablet hold-out results: training_data/training_report.json")


## 4. Check an image


In [ ]:
def predict_image(image_path):
    with Image.open(image_path) as image:
        probabilities = predict(transform(image.convert("RGB")).unsqueeze(0))[0]
    from cnn import classify_probabilities
    return classify_probabilities(probabilities.unsqueeze(0), class_names, model.inference_policy)[0]

image_path = 'data/images (4).jpg'
label, score = predict_image(image_path)
print(f"{label}: model score {score:.1%}")


## 5. Camera feed — Glide350 prototype

Stop the previous feed, restart the kernel, and run the camera cell below. It loads the local glider_model.pth and current code directly. Keep the camera still and the scene clear during setup.

The model was trained on CAD renderings, not photographs of a physical glider. Its real-camera and airborne performance are unverified. Only accepted glider predictions get a box and X/Y coordinates. Background and uncertain predictions are hidden.

The earlier optional accuracy/image cells evaluate the previous photo classifier; their results do not measure this Glide350 prototype. Use Stop/Interrupt to finish the camera feed.


In [ ]:
from pathlib import Path
import importlib.util
import sys

PROJECT = Path('.')

# Load the current local files directly, even if an old module is cached.
def _load_camera_module(name, filename):
    spec = importlib.util.spec_from_file_location(name, PROJECT / filename)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module

_camera_cnn = _load_camera_module("cnn", "cnn.py")
_camera_tracker = _load_camera_module("trackinh", "trackinh.py")
_camera_model, _camera_transform, _camera_names = _camera_cnn.load_model(PROJECT / 'glider_model.pth')

def _classify_camera_frames(frames):
    return _camera_cnn.predict_frames(frames, _camera_model, _camera_transform, _camera_names)

print(f"Camera v{_camera_tracker.TRACKER_VERSION} | maximum {_camera_tracker.MAX_REGIONS} boxes", flush=True)
print(f"Tracker: {_camera_tracker.__file__}", flush=True)
print("Glide350 prototype: trained on rendered views; physical-camera recognition is unverified.", flush=True)
_camera_tracker.run_camera(classify_frames=_classify_camera_frames, camera_index=0, inline=True, target_label="glider")
